1. **Nome(s):** Leonardo Silveira Kiepper

2. **Matrícula(s):** 2026140417

3. **E-mail(s) institucional(is):** leonardo.kiepper@edu.ufes.br

3. **Curso(s):** Doutorado em Ciência da Computação

4. **Link do seu Colab com suas soluções (para que o professor possa acessar e rodar o código):** https://colab.research.google.com/github/LeoKiepper/Ciencia-de-Dados/blob/main/trabalho1/T1_leonardo_kiepper.ipynb


# Introdução

Cada compra que uma pessoa faz gera dados que são úteis para diversas aplicações práticas. Por exemplo, em se tratando de artigos de mercearia ou supermercado, ter um registro dos preços de produtos de interesse permite o monitoramento de gastos pessoais, precificação de receitas, etc. De modo geral, para tais aplicações serem viáveis em escala pessoal, é fundamental que a solução seja de baixo custo e de baixa manutenção, o que elimina soluções como controle de estoque informatizado, que muitos supermercados usam.

Por exigência de lei, quando requisitado, a maioria dos estabelecimentos já entrega o cupom fiscal detalhando a compra. Sendo assim, o cupom fiscal é o artefato físico natural de onde extrair os dados necessários para estas aplicaçãoes. No entanto, a tarefa de ler cupons fiscais e extrair esses dados manualmente não é prática, devido ao volume de dados necessário para que a solução implementada tenha utilidade.

De modo geral, os dados importantes para tais aplicações, que estariam presentes em uma nota ou cupom fiscal impresso, não são prontamente utilizáveis. Além da etapa de digitalização para converter em arquivo digital, e obrigatoriamente o reconhecimento óptico de caracteres (OCR) para converter imagem para texto, é necessário uma passada de extração de dados estruturada, na qual dados como descrição do produto e valor, são individualizados num arquivo tabular. Os dados mínimos de interesse são a descrição do produto, e o valor.

Mesmo tendo feito uma leitura e extração estruturada perfeitas, isto é, de modo que os dados constando no cupom fiscal físico tenham sido copiados exatamente para o arquivo digital, os dados ainda precisam ser curados, especificamente a descrição do produto. O desafio aqui é que a descrição do produto constando num cupom fiscal é extremamente truncada, para caber na dimensão limitada do papel térmico que normalmente é utilizado para esse propósito. Soma-se a isso os fatos de que cada comércio descreve e abrevia o produto de sua forma particular, e que, depois de identificado o produto a partir da descrição, é necessário adotar alguma forma padronizada de indexar o produto identificado para que a solução implementada seja amplamente aplicável a situações reais, e fica evidente que o problema não é trivial.

Limitando o escopo do projeto a predizer a descrição completa a partir da descrição truncada, procurou-se um conjunto de dados que suportasse essa tarefa. A base de dados aberta ELEVEN (https://github.com/vinidiol/descmerc) traz aproximadamente 1,1 milhão de linhas retiradas de nota fiscais anonimizadas, constando: a descrição do produto truncada, um código na Nomenclatura Comum do Mercosul (NCM) declarado pelo emissor, e um rótulo anotado por auditores fiscais especialistas. 

In [48]:
%pip install tensorflow
%pip install itables

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [49]:
#@title Carrega o dataset

import os
import pandas as pd
from langdetect import detect, DetectorFactory, LangDetectException
from itables import show
# Carrega a partir do github
base = pd.read_csv('https://github.com/LeoKiepper/Ciencia-de-Dados/raw/main/trabalho2/descmerc/BaseDESC_NCM.zip',
                   sep='\t', dtype={'XPROD': str})

show(base)


In [50]:
#@title Amostra uma fatia do dataset
import numpy as np

# A título de ensaio, pega uma fatia pequena, e fixa a semente com random_state
# semente = 2003343720
semente = np.random.randint(0, 2**31)
amostra=base.sample(n=1000, random_state=semente)
amostra.drop(inplace=True, columns=[col for col in amostra.columns if col not in ['XPROD','NCM','rotulo']])
show(amostra)

# Metodogia

Foi treinada uma CNN para exprimir relações entre a descrição do produto (coluna "XPROD" do *dataset*, entrada da rede) e o código NCM (saída, ou predição, da rede). A coluna "XPROD" tem dados formatados como *string*. Para tratar a imprevisibilidade do estilo de escrita de cada descrição, foi definida a função de preprocessamento abaixo.

A lógica por trás dessa função é remover ruído das descrições que não contribuiriam para descrever o produto, tomando cuidado para não remover sinal junto com o ruído, levando em conta características específicas de como pessoas escrevem essas descrições. Duas regras importantes implementadas:

>"Qualificadores que sejam aplicáveis a múltiplos produtos de múltiplas categorias não contribuem para individualizar o produto para uma categoria, então são ruído". (Por exemplo, características da embalagem, marketing como 'leve X pague Y', quantidades)

>"Se há risco de eliminar sinal, não remover. A informação passa a ser dado para a rede aprender ou ignorar."

Um caso notável de qualificadores que foram mantidos pela segunda regra são cores. Para alguns produtos, a cor é um qualificador que não muda a função. Por exemplo, uma escova de dente pode ser de qualquer cor, e ter exatamente a mesma função. Para outros, a cor é um qualificador que distingue o produto de outros semelhantes. Por exemplo, vinho branco é diferente de vinho tinto.

Com base nessas duas regras, foi criada uma sequência de regex para filtrar as strings. Esses regex removem grafias observadas para quantidades (que em geral não seguem grafia padronizada de unidades de medida, como 'metro' → 'm', 'litro' → 'l', 'grama' → 'g, etc). Depois de filtradas, as strings são tokenizadas. Alguns problemas são atacados em mais de uma etapa, para retirá-los de formas diferentes que não são fáceis de pegar por uma regra única. São feitas as seguintes operações. Para listagens completas, vide o código abaixo.

1. Remover acentos e quantidades/tamanhos de embalagem
2. Dividir o texto em tokens
3. Converter o texto para letras minúsculas
4. Separar tokens que usam '.' ou '/' como separador
5. Remover símbolos de pontuação de cada token e descartar tokens vazios
6. Remover unidades, embalagens, marketing, tamanho, referência e stop words
7. Remover letras soltas (tokens com apenas uma letra)
8. Remover números soltos (3, 1/2, 030003, 20x25) e quantidades (750ml, 1kg)
9. Remover códigos de referência (REF 123, C21, SG-8407, 27814E)


In [51]:
#@title Tokeniza o dataset
import re, unicodedata
import pandas as pd
import nltk
import string
from nltk import word_tokenize
from nltk.corpus import stopwords
from itables import show
from collections import Counter
from functools import lru_cache

nltk.download('punkt_tab')
nltk.download('stopwords')

NUM  = r'\d+(?:[.,]\d+)?'
UNID = r'(?:M2|MLN|ML|MLS|M|MT|MTS|G|GR|GRS|KG|KGS|L|LT|LTS|LI|LITROS?|UN|UND|UNDS|UNID|U|PC|PCS|KIT)'
EMB  = r'(?:CX|CXA|UN|UND|PT|PC|FD|DP|SC|CJ|PCT|BD|FR|GA|GF|DZ)'
FIM  = r'\.?(?![A-Z\d])'          # fim do token: não seguido de letra ou dígito
PADRAO_QTD = re.compile(rf'''
    (?<![\w,.]){EMB}?(?:{NUM}\s?X\s?)+{NUM}\s?{UNID}{FIM}   # 12X500ML, DP12X24X22G
  | (?<![\w,.]){EMB}(?:{NUM}\s?X\s?)+{NUM}{FIM}             # CX20X50
  | (?<![\w,.])1\s?X\s?\d+{FIM}                             # 1 X 12
  | (?<![\w,.]){EMB}?{NUM}\s?{UNID}{FIM}                    # 500ML, 1,5 L, CX10KG
  | \b{EMB}\s?[/-]\s?\d+(?:\s?[/-]\s?{EMB})?{FIM}           # CX/12, UN/0001, CX-6-UN
  | \b{EMB}\d+{EMB}?{FIM}                                   # PT0001UN
  | \bC/\s?\d+(?:\s?{UNID})?{FIM}                           # C/12, C/20MTS
  | \bQTD\.?\s?{NUM}(?:\s{EMB})?                            # QTD. 2.00 DP
  | \(\s?\d+\s?{UNID}?\s?\)                                 # (12 UN)
  | \(CX[\d,]+(?:M2|PCS?)\)                                 # (CX1,62M2)
  | -\s?\d+VL\b                                             # - 24VL
  | \bF\.?P\.?S\.?\s?-?\s?\d+\+?                            # FPS 70
  | \bLV?\d+\s?PG?\d+(?:ML?|G|GR)?\b                        # LV12PG11, L4P3, LV16 PG15
  | \b\d+\s?{EMB}\s?\d+\s?{UNID}{FIM}                       # 1CX12UN
''', re.X)
ALFANUM = re.compile(r'(?=.*\d)(?=.*[a-z])[a-z\d]+')   # tem letra e dígito
FRAG    = re.compile(r'[a-z]{3,}')                      # fragmento alfabético resgatável

# '.' e '/' usados como separador (não quebra decimais e frações: 2.5mm, 1/2)
SEP = re.compile(r'(?<!\d)[./]|[./](?!\d)')
TABELA_PONT = str.maketrans('', '', string.punctuation)
# número+unidade dentro de token (CAB.SAV.750ML -> 750ml, 1-KG -> 1kg, 12x40g)
QTD_TOKEN = re.compile(r'(?:\d+x)*\d+(?:m2|mln|ml|mls|m|mt|mts|g|gr|grs|kg|kgs|l|lt|lts|li|un|und|unid|pc|pcs)')

# tokens só de dígitos, separadores e 'x': 3, 1/2, 030003, 20x25
NUMERICO = re.compile(r'[\d.,/+\-%*:;"\'&#x]*\d[\d.,/+\-%*:;"\'&#x]*')
# códigos de referência (preserva DN15, RJ11, IP65, E27, MM2, CA50 e medidas como 2700K, 6800W)
CODIGO = re.compile(r'(?:ref|cod|mva|lote|venc|nr|n\.)[.:\-]?\w[\w.\-/]*'          # REF 123, MVA:29
                    r'|[a-z]-?\d+[a-z]?'                                          # C21, R0091, S08
                    r'|[a-z]-\d+/\d+'                                             # B-4472/17
                    r'|\d{4,}-?(?!(?:k|w|v|a|kv|ka|va|mm|cm|m)$)[a-z]{1,2}'        # 27814E, 1328-M
                    r'|[a-z]{1,2}\d{4,}'                                          # MK2431
                    r'|r\.\d+|a/\d+|tam\d+|\d+h\d+'                               # R.123, A/4552, TAM2, 1H15
                    r'|(?!dn|rj|ip|mm|ca|e\d)[a-z]{2}-?\d+[a-z]?')                # SG-8407, GL83, BR-01
# dimensões NxM(xP), com ou sem unidade: 20x25, 67x370mm, 32mmx22mm, 2750x1840x15mm, 19mmx10m
DIMENSAO = re.compile(r'(?:\d+(?:mm|cm|mts?|m)?x)+\d+(?:mm|cm|mts?|m)?')

def _sem_acento(s): return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()

RUIDO = (
    {'ml','mls','gr','grs','kg','kgs','lt','lts','un','und','unid','cx','cxa','fd','dp','pct','pc','pcs',
     'mt','mts','pt','fr','sc','bd','gf'}                                         # unidades e embalagens
  | {'pack','sache','refil','cartela','pet','lata'}                               # embalagem
  | {'premium','prem','especial','classic','classico','class','super','ultra','plus','max','maxi','top',
     'gold','golden','star','prime','master','power','smart','easy','life','ideal','royal','imperial',
     'new','novo','nova','tradic','orig','duo','one'}                             # marketing / linha
  | {'grande','gde','pequeno','pequena','peq'}                                    # tamanho
  | {'ref','cod','nr'}                                                            # referência
  | ({_sem_acento(w) for w in stopwords.words('portuguese')} - {'com','sem','nao'})
)

@lru_cache(maxsize=None)
def _tokeniza(df: pd.Series):
    # 1. Remover acentos e quantidades/tamanhos de embalagem (sobre a string)
    texto = PADRAO_QTD.sub(' ', _sem_acento(df).upper())

    # 2. Dividir o texto em tokens
    tokens = word_tokenize(texto)

    # 3. Converter o texto para letras minúsculas
    tokens = [w.lower() for w in tokens]

    # 4. Separar tokens que usam '.' ou '/' como separador (VIN.PORT.CHAMINE, P/TRAT)
    tokens = [p for w in tokens for p in SEP.split(w)]

    # 5. Remover símbolos de pontuação de cada token e descartar tokens vazios
    tokens = [w.translate(TABELA_PONT) for w in tokens]
    tokens = [w for w in tokens if w]

    # 6. Remover unidades, embalagens, marketing, tamanho, referência e stop words
    tokens = [w for w in tokens if w not in RUIDO]

    # 7. Remover letras soltas
    tokens = [w for w in tokens if not (len(w) == 1 and w.isalpha())]

    # 8. Remover números soltos (3, 1/2, 030003), quantidades (750ml, 1kg) e dimensões (67x370mm)
    tokens = [w for w in tokens if not NUMERICO.fullmatch(w)
                               and not QTD_TOKEN.fullmatch(w)
                               and not DIMENSAO.fullmatch(w)]

    # 9. Remover códigos de referência (REF 123, C21, SG-8407, 27814E)
    tokens = [w for w in tokens if not CODIGO.fullmatch(w)]

    return tokens
  
# Em quantas descrições da base cada token aparece (calculado uma vez)
DF_TOKENS = Counter(t for toks in base['XPROD'].map(_tokeniza) for t in set(toks))

def resgata(tokens):
    """Alfanuméricos que aparecem em no máximo uma descrição da base são trocados
    pelos seus fragmentos alfabéticos (>= 3 letras, fora do RUIDO); sem fragmento, somem."""
    out = []
    for w in tokens:
        if DF_TOKENS[w] <= 1 and ALFANUM.fullmatch(w):
            out += [p for p in FRAG.findall(w) if p not in RUIDO]
        else:
            out.append(w)
    return out

def preprocessamento(df: pd.Series):
    # Tokeniza e resgata fragmentos de alfanuméricos únicos
    return resgata(_tokeniza(df))

show(amostra['XPROD'].apply(preprocessamento))


[nltk_data] Downloading package punkt_tab to /home/Leo/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/Leo/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [52]:
def calculaX(x: pd.Series):
    """TF-IDF de x. Devolve (M, vocab): M é CSR float32 (linhas na ordem de x), vocab é o array de tokens."""
    vectorizer = TfidfVectorizer(analyzer=preprocessamento, dtype=np.float32)
    M = vectorizer.fit_transform(x)
    return M, vectorizer.get_feature_names_out()

In [54]:
#@title Calcula ou carrega a matriz TF-IDF do dataset completo
import os, io, urllib.request
import scipy.sparse as sp

REPO_RAW  = 'https://raw.githubusercontent.com/LeoKiepper/Ciencia-de-Dados/main/trabalho2/'
ARQ_X     = 'X_tfidf.npz'
ARQ_VOCAB = 'X_tfidf_vocab.csv'

def carregaX(x: pd.Series):
    """Carrega (M, vocab): arquivo local -> GitHub -> calcula com calculaX e salva."""
    if os.path.exists(ARQ_X) and os.path.exists(ARQ_VOCAB):
        return sp.load_npz(ARQ_X), pd.read_csv(ARQ_VOCAB, keep_default_na=False)['token'].to_numpy()
    try:
        M = sp.load_npz(io.BytesIO(urllib.request.urlopen(REPO_RAW + ARQ_X).read()))
        return M, pd.read_csv(REPO_RAW + ARQ_VOCAB, keep_default_na=False)['token'].to_numpy()
    except Exception:
        pass
    M, vocab = calculaX(x)
    sp.save_npz(ARQ_X, M)
    pd.Series(vocab, name='token').to_csv(ARQ_VOCAB, index=False)
    print(f'Matriz calculada e salva em {ARQ_X} e {ARQ_VOCAB}')
    return M, vocab

def comoDataFrame(M, vocab, indice=None, linhas=slice(0, 20), so_nao_nulas=True):
    """Fatia de M como DataFrame, para visualização. `linhas`: slice ou lista de posições.
    Com so_nao_nulas=True, mostra só as colunas com algum valor nas linhas escolhidas."""
    sub = M[linhas]
    idx = None if indice is None else pd.Index(indice)[linhas]
    cols = np.unique(sub.indices) if so_nao_nulas else np.arange(M.shape[1])
    return pd.DataFrame(sub[:, cols].toarray(), index=idx, columns=np.asarray(vocab)[cols])

X, vocab = carregaX(base['XPROD'])
print(f'X {X.shape} {X.dtype}, {X.nnz} valores não nulos')
show(comoDataFrame(X, vocab, indice=base.index))
print(pd.Series(vocab).sample(50).tolist())


X (1117623, 85390) float32, 4937310 valores não nulos


['conformado', '18c', 'elasico', 'trag', 'gtp114', 'kp', 'rolafacil', 'gbz', '120gm', 'marselha', 'p700mm', '3dob', 'baccarat', '01paral', 'enrolar', 'showy', '1vmu', 'simasbala', 'beja', 'xii', 'mabnt', 'foamspray', '1pgo50x15', '100f', 'desengordurado', 'horiz', 'camara', 'abastecer', 'cp250gcx24', 'tk', 'especif', 'ressecada', 'floresgmezn', 'kanto', 'isw', 'manch', 'balancedv', 'zanini', 'gtp1', 'levrest', 'triolade', '305cm', 'assadeira', '34toralf', 'dpc', 'carvlho', 'fametraontina', 'nts', 'ligua', 'martillac']
